# Plot Retrieval Results

Set the paths in the first cell, then run the notebook top to bottom.

In [ ]:
from pathlib import Path
from io import StringIO

import numpy as np
import matplotlib.pyplot as plt
from corner import corner

from floppity import Retrieval, RetrievalOutput, helpers

OUTPUT_DIR = Path("output_FlopPITy")
RETRIEVAL_PATH = OUTPUT_DIR / "retrieval.pkl"
ROUND_INDEX = -1
N_CORNER_SAMPLES = 5000
MAX_SPECTRA_TO_DRAW = 300
ATMOSPHERE_HEADER = ["T", "P", "H2O", "CH4", "NH3", "PH3", "H2S", "H2", "He"]

In [ ]:
def round_data_path(output_dir, round_index):
    round_dirs = sorted((output_dir / "rounds").glob("round_*"))
    if not round_dirs:
        raise FileNotFoundError(f"No round directories found in {output_dir / 'rounds'}")
    if round_index < 0:
        round_dir = round_dirs[round_index]
    else:
        round_dir = output_dir / "rounds" / f"round_{round_index:03d}"
    return round_dir / "training_data.npz"


def atmosphere_path(output_dir, round_index):
    if round_index < 0:
        paths = sorted(output_dir.glob("mixingratios_round_*.dat"))
        if not paths:
            raise FileNotFoundError(f"No mixingratios_round_*.dat files found in {output_dir}")
        return paths[round_index]
    return output_dir / f"mixingratios_round_{round_index}.dat"


def proposal_for_corner(retrieval, alpha):
    if alpha > 0:
        if getattr(retrieval, "posteriors", None):
            return retrieval.posteriors[-1]
        proposal = retrieval.proposals[-1]
        if hasattr(proposal, "posterior"):
            return proposal.posterior
    return retrieval.proposals[-1]


def proposal_sample_mask(round_data, alpha):
    sources = round_data.get("sample_sources")
    if alpha <= 0:
        return slice(None)
    if sources is None:
        raise ValueError("This run used alpha > 0, but the round archive has no sample_sources. Re-run with save_data=True using the current FlopPITy branch.")
    return sources == "proposal"


def sample_rows(n_rows, max_rows):
    if n_rows <= max_rows:
        return np.arange(n_rows)
    rng = np.random.default_rng(0)
    return np.sort(rng.choice(n_rows, max_rows, replace=False))

In [ ]:
R = Retrieval.load(RETRIEVAL_PATH)
alpha = float(getattr(R, "alpha", 0))
data_path = round_data_path(OUTPUT_DIR, ROUND_INDEX)
round_data = RetrievalOutput.load_round_data(data_path)
spectra = round_data.get("post_spec", round_data["spec"])
mask = proposal_sample_mask(round_data, alpha)

print(f"Loaded retrieval: {RETRIEVAL_PATH}")
print(f"Loaded round data: {data_path}")
print(f"alpha = {alpha}")
if isinstance(mask, np.ndarray):
    print(f"Using {mask.sum()} proposal-sampled spectra out of {len(mask)} total samples")

## Corner Plot

In [ ]:
proposal = proposal_for_corner(R, alpha)
samples = proposal.sample((N_CORNER_SAMPLES,)).detach().cpu().numpy()
samples = samples.reshape(-1, len(R.parameters))
samples_physical = helpers.convert_cube(samples, R.parameters)

fig = corner(samples_physical, labels=list(R.parameters.keys()), show_titles=True)
fig.suptitle("Uninflated posterior" if alpha > 0 else "Posterior", y=1.02)
plt.show()

## Retrieved Spectra

In [ ]:
obs_items = list(R.obs.items())
fig, axes = plt.subplots(len(obs_items), 1, figsize=(9, 3.2 * len(obs_items)), squeeze=False)

for ax, (obs_key, obs) in zip(axes[:, 0], obs_items):
    spec = spectra[obs_key][mask]
    rows = sample_rows(len(spec), MAX_SPECTRA_TO_DRAW)
    for row in rows:
        ax.plot(obs[:, 0], spec[row], color="tab:blue", alpha=0.04, lw=0.8)
    median = np.nanmedian(spec, axis=0)
    lo, hi = np.nanpercentile(spec, [16, 84], axis=0)
    ax.fill_between(obs[:, 0], lo, hi, color="tab:blue", alpha=0.18, lw=0)
    ax.plot(obs[:, 0], median, color="tab:blue", lw=2, label="median model")
    ax.errorbar(obs[:, 0], obs[:, 1], yerr=obs[:, 2], fmt="o", ms=3, color="black", label="data")
    ax.set_title(str(obs_key))
    ax.set_xlabel("Wavelength")
    ax.set_ylabel("Flux / depth")
    ax.legend()

fig.tight_layout()
plt.show()

## Atmospheric Structures

In [ ]:
def parse_mixingratios(path, fallback_header):
    blocks = []
    current = []
    for line in Path(path).read_text().splitlines():
        if line.startswith("# round=") and current:
            blocks.append(current)
            current = []
        current.append(line)
    if current:
        blocks.append(current)

    profiles = []
    for block in blocks:
        header = None
        numeric = []
        for line in block:
            stripped = line.strip()
            if not stripped:
                continue
            if stripped.startswith("#"):
                candidate = stripped.lstrip("#").split()
                if len(candidate) >= 2 and candidate[0].startswith("T") and candidate[1].startswith("P"):
                    header = [name.split("[")[0] for name in candidate]
                continue
            numeric.append(stripped)
        if not numeric:
            continue
        arr = np.loadtxt(StringIO("\n".join(numeric)))
        arr = np.atleast_2d(arr)
        names = header or fallback_header[: arr.shape[1]]
        profiles.append({name: arr[:, i] for i, name in enumerate(names)})
    return profiles


def profile_grid(profiles, value_name, log_value=False, n_grid=200):
    ranges = [(np.nanmin(p["P"]), np.nanmax(p["P"])) for p in profiles if value_name in p]
    p_min = max(low for low, high in ranges)
    p_max = min(high for low, high in ranges)
    pressure = np.geomspace(p_min, p_max, n_grid)
    log_pressure = np.log10(pressure)
    values = []
    for profile in profiles:
        if value_name not in profile:
            continue
        p = np.asarray(profile["P"])
        v = np.asarray(profile[value_name])
        order = np.argsort(p)
        y = np.log10(np.clip(v[order], 1e-300, None)) if log_value else v[order]
        values.append(np.interp(log_pressure, np.log10(p[order]), y))
    return pressure, np.asarray(values)


def plot_profile_envelope(ax, pressure, values, label, xlabel):
    bands = [(0.135, 99.865, 0.12), (2.275, 97.725, 0.18), (15.865, 84.135, 0.28)]
    for lo_q, hi_q, alpha_fill in bands:
        lo, hi = np.nanpercentile(values, [lo_q, hi_q], axis=0)
        ax.fill_betweenx(pressure, lo, hi, alpha=alpha_fill, color="tab:blue", lw=0)
    ax.plot(np.nanmedian(values, axis=0), pressure, color="tab:blue", lw=2, label=label)
    ax.set_yscale("log")
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Pressure [bar]")
    ax.legend()


atm_path = atmosphere_path(OUTPUT_DIR, ROUND_INDEX)
profiles = parse_mixingratios(atm_path, ATMOSPHERE_HEADER)
print(f"Loaded {len(profiles)} atmospheric profiles from {atm_path}")

## Temperature Structure

In [ ]:
pressure, temperature = profile_grid(profiles, "T", log_value=False)
fig, ax = plt.subplots(figsize=(5, 6))
plot_profile_envelope(ax, pressure, temperature, "median T", "Temperature [K]")
fig.tight_layout()
plt.show()

## Molecular Abundances

In [ ]:
species = [name for name in profiles[0] if name not in {"T", "P"}]
ncols = 2
nrows = int(np.ceil(len(species) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(10, 4 * nrows), squeeze=False)

for ax, species_name in zip(axes.ravel(), species):
    pressure, abundance = profile_grid(profiles, species_name, log_value=True)
    plot_profile_envelope(ax, pressure, abundance, species_name, "log10 abundance")
    ax.set_title(species_name)

for ax in axes.ravel()[len(species):]:
    ax.axis("off")

fig.tight_layout()
plt.show()